# Python Advanced - Day 7 Homework

**Runbook alignment (Day 7, Hours 25-28):**
- Hour 25: From objects to tables: schema thinking + repository idea
- Hour 26: sqlite3 CRUD with parameterized queries
- Hour 27: Row mapping: converting rows to objects
- Hour 28: Transactions + integrity + initialization patterns

This homework moves the tracker project from in-memory objects to sqlite persistence with repositories, mappers, and transactions.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day7.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- If you stage work in `submissions/`, keep exactly one real `*Advanced_Day7*.ipynb` there; the bundled sample notebook is only a fallback for local validation.
- If you stage work in `submissions/`, keep exactly one real `*Advanced_Day7*.ipynb` there; the bundled sample notebook is only a fallback for local validation.
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 25 - From objects to tables: schema thinking + repository idea

**Goal:** Build a table design and repository boundary for tracker records.

**Best practice:** Design the table shape from the domain model before writing CRUD code.

**Avoid this pitfall:** Letting ad hoc columns accumulate without a clear model makes mapping harder.


In [ ]:
import sqlite3
from dataclasses import dataclass

@dataclass
class TaskRecord:
    id: str
    title: str
    priority: str
    done: int

class TaskRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection

    def initialize(self) -> None:
        self.connection.execute(
            "CREATE TABLE IF NOT EXISTS tasks (id TEXT PRIMARY KEY, title TEXT NOT NULL, priority TEXT NOT NULL, done INTEGER NOT NULL)"
        )
        self.connection.commit()

conn = sqlite3.connect(":memory:")
repository = TaskRepository(conn)
repository.initialize()

print("Hour 25 | table name: tasks")
print("Hour 25 | primary key: id")
print("Hour 25 | columns: title, priority, done")
print("Hour 25 | repository role: isolate sql details")
print("Hour 25 | schema rule: one object maps to one row shape")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 26 - sqlite3 CRUD with parameterized queries

**Goal:** Build basic sqlite CRUD functions using parameters safely.

**Best practice:** Use parameterized queries for every write and lookup.

**Avoid this pitfall:** String-building SQL by hand invites bugs and injection issues.


In [ ]:
conn.execute("DELETE FROM tasks")
conn.executemany(
    "INSERT INTO tasks (id, title, priority, done) VALUES (?, ?, ?, ?)",
    [
        ("task-101", "Ship report", "high", 0),
        ("task-102", "Review notes", "medium", 0),
    ],
)
conn.commit()
selected_count = conn.execute("SELECT COUNT(*) FROM tasks").fetchone()[0]
updated_rows = conn.execute(
    "UPDATE tasks SET priority = ? WHERE id = ?",
    ("medium", "task-101"),
).rowcount
deleted_rows = conn.execute(
    "DELETE FROM tasks WHERE id = ?",
    ("task-102",),
).rowcount
conn.commit()

print("Hour 26 | insert result: task-101 saved")
print(f"Hour 26 | select result count: {selected_count}")
print(f"Hour 26 | update result: {updated_rows} row changed")
print(f"Hour 26 | delete result: {deleted_rows} row removed")
print("Hour 26 | sql rule: parameterize every query")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 3: Hour 27 - Row mapping: converting rows to objects

**Goal:** Build a mapper that turns sqlite rows into tracker objects.

**Best practice:** Convert database rows to domain objects in one consistent place.

**Avoid this pitfall:** Leaking raw tuples upward forces the rest of the app to know DB details.


In [ ]:
import sqlite3
from dataclasses import dataclass

conn.row_factory = sqlite3.Row
conn.execute(
    "INSERT OR REPLACE INTO tasks (id, title, priority, done) VALUES (?, ?, ?, ?)",
    ("task-103", "Plan demo", "low", 0),
)
conn.commit()

@dataclass
class Task:
    id: str
    title: str

def map_row(row: sqlite3.Row | None) -> Task | None:
    if row is None:
        return None
    return Task(id=row["id"], title=row["title"])

first_row = conn.execute(
    "SELECT id, title FROM tasks WHERE id = ?",
    ("task-101",),
).fetchone()
missing_row = conn.execute(
    "SELECT id, title FROM tasks WHERE id = ?",
    ("task-404",),
).fetchone()
mapped_tasks = [
    map_row(row)
    for row in conn.execute("SELECT id, title FROM tasks ORDER BY id").fetchall()
]

print(f"Hour 27 | row type: {type(first_row).__module__}.{type(first_row).__name__}")
print(f"Hour 27 | mapped object: {map_row(first_row)}")
print(f"Hour 27 | missing row: {map_row(missing_row)}")
print(f"Hour 27 | list mapped count: {len(mapped_tasks)}")
print("Hour 27 | mapping rule: convert rows at the repository boundary")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 4: Hour 28 - Transactions + integrity + initialization patterns

**Goal:** Build an initialization and transaction flow that protects data integrity.

**Best practice:** Initialize schema early and group related writes inside a transaction.

**Avoid this pitfall:** Partially applied writes leave the database in an inconsistent state.


In [ ]:
tx_conn = sqlite3.connect(":memory:")
tx_conn.execute(
    "CREATE TABLE tasks (id TEXT PRIMARY KEY, title TEXT NOT NULL, priority TEXT NOT NULL, done INTEGER NOT NULL)"
)
tx_conn.commit()

rollback_happened = False
try:
    tx_conn.execute("BEGIN")
    tx_conn.execute(
        "INSERT INTO tasks (id, title, priority, done) VALUES (?, ?, ?, ?)",
        ("task-201", "Stage release", "high", 0),
    )
    tx_conn.execute(
        "INSERT INTO tasks (id, title, priority, done) VALUES (?, ?, ?, ?)",
        ("task-201", "Duplicate id", "high", 0),
    )
    tx_conn.commit()
except sqlite3.IntegrityError:
    tx_conn.rollback()
    rollback_happened = True

tx_conn.execute("BEGIN")
tx_conn.execute(
    "INSERT INTO tasks (id, title, priority, done) VALUES (?, ?, ?, ?)",
    ("task-202", "Ship checklist", "medium", 0),
)
tx_conn.commit()

print("Hour 28 | init script: schema ready")
print("Hour 28 | transaction begin: ok")
print(f"Hour 28 | rollback on error: {'yes' if rollback_happened else 'no'}")
print("Hour 28 | commit result: saved")
print("Hour 28 | integrity rule: keep related writes together")
tx_conn.close()
conn.close()


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day7_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


## Autograder Helper - Canonical Output Reference (Required Graded Tasks)

Use the reference below to understand the kinds of canonical markers the local Advanced Day 7 configuration checks for. Keep your own solution authentic, but make sure equivalent required lines appear when your notebook is converted to `advanced_day7.py`.

```python
# Canonical script for Advanced Day 7 grading by autograder (advanced_day7.py)
def main():
    print('Hour 25 | table name: tasks')
    print('Hour 25 | primary key: id')
    print('Hour 25 | columns: title, priority, done')
    print('Hour 25 | repository role: isolate sql details')
    print('Hour 25 | schema rule: one object maps to one row shape')
    print('Hour 26 | insert result: task-101 saved')
    print('Hour 26 | select result count: 2')
    print('Hour 26 | update result: 1 row changed')
    print('Hour 26 | delete result: 1 row removed')
    print('Hour 26 | sql rule: parameterize every query')
    print('Hour 27 | row type: sqlite3.Row')
    print("Hour 27 | mapped object: Task(id='task-101', title='Ship report')")
    print('Hour 27 | missing row: None')
    print('Hour 27 | list mapped count: 2')
    print('Hour 27 | mapping rule: convert rows at the repository boundary')
    print('Hour 28 | init script: schema ready')
    print('Hour 28 | transaction begin: ok')
    print('Hour 28 | rollback on error: yes')
    print('Hour 28 | commit result: saved')
    print('Hour 28 | integrity rule: keep related writes together')

if __name__ == "__main__":
    main()
```
